# HCDE 530 — A5: Pandas Analysis

### Instructions
In class you used pandas to answer five questions about a dataset. For A5 you are doing that on your own data — the dataset you chose for Mini Project 1 — and documenting what you found.

This assignment is closely connected to MP1. The analysis you do here is the foundation for the notebook you will build in Week 6.

### Introduction
talk about the data here and also mention the analysis questions 

---
### 0. Setting up Pandas

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("pandas version:", pd.__version__)

pandas version: 3.0.2


---
### 1. Importing up the API

To begin answering my data analysis questions, I need to set up endpoints for 3 different APIs, each dealing with a different type of player information. This includes one to fetch summoner ID data (where I'll query for Player Universally Unique IDs (PUUID)), one to extract game match information based on those PUUIDs, and finally an API to extract champion picks per match. 

##### 1a. Player Universally Unique IDs

In [7]:
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path

#function to load .env file so that this code can read the API key on it 
def _load_env(path: Path) -> None:
    """Load KEY=VALUE pairs from a .env file into os.environ (does not override existing vars)."""
    if not path.is_file():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, val)


# Resolve Week 5/.env whether the kernel cwd is the repo root or the Week 5 folder
for _env in (Path(".env"), Path("Week 5") / ".env"):
    _load_env(_env)

api_key = os.environ.get("RIOT_API_KEY", "").strip() #the api key I stored in .env, will be regenerated every 24 hours
if not api_key:
    raise ValueError("RIOT_API_KEY missing: add it to Week 5/.env or set it in the environment.")

base = "https://na1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/IV" #endpoint for summoner ID data, need to query for PUUIDs later
# Riot sits behind Cloudflare; default User-Agent "Python-urllib/..." often gets HTTP 403 + "error code: 1010".
headers = {
    "X-Riot-Token": api_key,
    "User-Agent": "HCDE530-A5/1.0 (pandas coursework; contact instructor)",
}

all_entries: list = []
for page in range(1, 326):  # pages 1 … 325
    url = f"{base}?page={page}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.load(resp)
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        raise RuntimeError(f"HTTP {e.code} {e.reason}: {body[:500]}") from e

    if not data:
        break
    all_entries.extend(data)
    time.sleep(2.0)

summoner_league_json = all_entries
print(len(summoner_league_json), "total entries across fetched pages")
if summoner_league_json:
    print("example keys:", list(summoner_league_json[0].keys()))

52469 total entries across fetched pages
example keys: ['leagueId', 'queueType', 'tier', 'rank', 'puuid', 'leaguePoints', 'wins', 'losses', 'veteran', 'inactive', 'freshBlood', 'hotStreak']


---
### 2. Turning JSON data into a DataFrame

In [ ]:
#Optional: pd.DataFrame(all_entries) after loop

##### 1b. Match ID

##### 1c. Champion Picks